# E0006 — Nemotron 3.5 Lightning / Kaggle Gate B

**Purpose:** one bounded TP=4 model load + one short local ARC-shaped generation.

Run this notebook **only after Gate A has been reviewed and passed**. It does not submit to ARC and does not read hidden answers.

Required Kaggle settings:
- Accelerator: **GPU L4 x4**
- Internet: **OFF**
- Attach the same Lightning checkpoint validated by Gate A
- Preferred checkpoint: `nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4`

Expected outputs:
- `/kaggle/working/e0006_gate_b_smoke.json`
- `/kaggle/working/e0006_gate_b_vllm.log`

The notebook performs one compatibility round only. Do not edit it into an open-ended tuning loop.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
OUT = Path("/kaggle/working/e0006_gate_b_smoke.json")
LOG = Path("/kaggle/working/e0006_gate_b_vllm.log")
MODEL_HINTS = ("nemotron", "lightning")
PORT = 8000
STARTUP_TIMEOUT_S = 1800
GPU_MEMORY_UTILIZATION = 0.88
MAX_MODEL_LEN = 8192
MAX_TOKENS = 64

PROMPT = """Find the common rule that maps each input grid to its output grid, then solve the test input. Return only the final grid.

Train 1
Input:
0 1
0 0
Output:
1 0
0 0

Train 2
Input:
0 0 1
0 0 0
Output:
1 0 0
0 0 0

Test Input:
0 0
1 0
"""

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

def request_json(url, payload=None, timeout=10):
    data = None
    headers = {}
    method = "GET"
    if payload is not None:
        data = json.dumps(payload).encode("utf-8")
        headers["Content-Type"] = "application/json"
        method = "POST"
    req = urllib.request.Request(url, data=data, headers=headers, method=method)
    with urllib.request.urlopen(req, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))

def discover_model_roots(root):
    if not root.exists():
        return []
    found = set()
    tokenizer_names = ("tokenizer.json", "tokenizer_config.json", "tokenizer.model")
    for config in root.rglob("config.json"):
        parent = config.parent
        low = str(parent).lower()
        if not any(h in low for h in MODEL_HINTS):
            continue
        if any((parent / name).exists() for name in tokenizer_names):
            found.add(parent)
    return sorted(found, key=lambda p: (0 if "nvfp4" in str(p).lower() else 1, len(str(p))))

def load_config(model_path):
    p = model_path / "config.json"
    if not p.exists():
        return {}
    try:
        obj = json.loads(p.read_text(encoding="utf-8"))
        return obj if isinstance(obj, dict) else {}
    except Exception:
        return {}

def infer_quantization(config):
    q = config.get("quantization_config")
    if not isinstance(q, dict):
        return None
    method = str(q.get("quant_method") or "").lower()
    algo = str(q.get("quant_algo") or "").lower()
    if "nvfp4" in algo or method in {"modelopt_fp4", "nvfp4"}:
        return "modelopt_fp4"
    return None

def gpu_snapshot():
    exe = shutil.which("nvidia-smi")
    if exe is None:
        return []
    cmd = [exe, "--query-gpu=index,name,memory.total,memory.used,memory.free,utilization.gpu",
           "--format=csv,noheader,nounits"]
    try:
        cp = subprocess.run(cmd, capture_output=True, text=True, timeout=15, check=True)
    except Exception:
        return []
    rows = []
    for line in cp.stdout.splitlines():
        f = [x.strip() for x in line.split(",")]
        if len(f) != 6:
            continue
        i, name, total, used, free, util = f
        rows.append({
            "index": int(i), "name": name,
            "memory_total_mib": int(total), "memory_used_mib": int(used),
            "memory_free_mib": int(free), "utilization_gpu_percent": int(util),
        })
    return rows

def tail_text(path, max_chars=20000):
    if not path.exists():
        return ""
    return path.read_text(encoding="utf-8", errors="replace")[-max_chars:]

def classify_failure(text, returncode):
    t = text.lower()
    if "out of memory" in t or "cuda oom" in t:
        return "OOM_LOAD_OR_INIT"
    if "no space left on device" in t:
        return "DISK_EXHAUSTED"
    if "no module named" in t or "command not found" in t:
        return "DEPENDENCY_MISSING"
    if "unrecognized arguments" in t or "no such option" in t:
        return "VLLM_VERSION_OR_FLAG_MISMATCH"
    if "quantization" in t and any(x in t for x in ("unsupported", "not supported", "unknown")):
        return "UNSUPPORTED_QUANTIZATION_PATH"
    if any(x in t for x in ("unsupported", "not implemented", "no kernel", "invalid device function")):
        return "UNSUPPORTED_KERNEL_OR_ARCH"
    if "model architecture" in t and "not supported" in t:
        return "UNSUPPORTED_MODEL_ARCH"
    if "address already in use" in t:
        return "PORT_IN_USE"
    if returncode is None:
        return "STARTUP_TIMEOUT"
    return "SERVER_EXITED"

def wait_for_health(base_url, proc, timeout_s):
    started = time.perf_counter()
    while time.perf_counter() - started < timeout_s:
        if proc.poll() is not None:
            return False, time.perf_counter() - started
        try:
            with urllib.request.urlopen(base_url + "/health", timeout=3) as r:
                if 200 <= r.status < 300:
                    return True, time.perf_counter() - started
        except Exception:
            pass
        time.sleep(5)
    return False, time.perf_counter() - started

report = {
    "experiment": "E0006",
    "gate": "B_TP4_ONE_GENERATION",
    "status": "STARTING",
    "purpose": "deployment_feasibility_only",
    "internet_required": False,
    "gpu_before": gpu_snapshot(),
}
proc = None

try:
    vllm = shutil.which("vllm")
    if vllm is None:
        raise RuntimeError("vllm executable not found in Kaggle image; Gate B must not install from internet")

    roots = discover_model_roots(INPUT_ROOT)
    if not roots:
        raise FileNotFoundError("No attached Nemotron/Lightning Hugging Face model root found under /kaggle/input")
    model = roots[0]
    cfg = load_config(model)
    quant = infer_quantization(cfg)

    cmd = [
        vllm, "serve", str(model),
        "--host", "127.0.0.1",
        "--port", str(PORT),
        "--tensor-parallel-size", "4",
        "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
        "--dtype", "bfloat16",
        "--mamba-ssm-cache-dtype", "float16",
        "--mamba-backend", "flashinfer",
        "--enable-mamba-cache-stochastic-rounding",
        "--mamba-cache-philox-rounds", "5",
        "--max-model-len", str(MAX_MODEL_LEN),
        "--enable-expert-parallel",
        "--reasoning-parser", "nemotron_v3",
        "--tool-call-parser", "qwen3_coder",
        "--enable-auto-tool-choice",
        "--enforce-eager",
    ]
    if quant:
        cmd += ["--quantization", quant]

    report["model_path"] = str(model)
    report["model_config_quantization"] = cfg.get("quantization_config")
    report["resolved_quantization"] = quant
    report["command"] = cmd
    report["vllm_executable"] = vllm

    LOG.parent.mkdir(parents=True, exist_ok=True)
    with LOG.open("w", encoding="utf-8") as fh:
        proc = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, text=True)

    base_url = f"http://127.0.0.1:{PORT}"
    healthy, startup_s = wait_for_health(base_url, proc, STARTUP_TIMEOUT_S)
    report["startup_seconds"] = startup_s
    report["gpu_after_startup"] = gpu_snapshot()

    if not healthy:
        tail = tail_text(LOG)
        rc = proc.poll()
        report.update({
            "status": "FAIL",
            "failure_class": classify_failure(tail, rc),
            "server_returncode": rc,
            "log_tail": tail,
        })
    else:
        models = request_json(base_url + "/v1/models", timeout=15)
        ids = [x.get("id") for x in models.get("data", []) if x.get("id")]
        if not ids:
            raise RuntimeError("vLLM health passed but /v1/models exposed no model id")
        served_model = ids[0]
        payload = {
            "model": served_model,
            "messages": [{"role": "user", "content": PROMPT}],
            "temperature": 0.0,
            "max_tokens": MAX_TOKENS,
        }
        t0 = time.perf_counter()
        response = request_json(base_url + "/v1/chat/completions", payload=payload, timeout=600)
        generation_s = time.perf_counter() - t0
        usage = response.get("usage") or {}
        choices = response.get("choices") or []
        msg = choices[0].get("message", {}) if choices else {}
        ct = usage.get("completion_tokens")
        report.update({
            "status": "PASS_GATE_B",
            "served_model": served_model,
            "generation_seconds": generation_s,
            "prompt_tokens": usage.get("prompt_tokens"),
            "completion_tokens": ct,
            "completion_tokens_per_second": (ct / generation_s if isinstance(ct, (int, float)) and generation_s > 0 else None),
            "output_text": msg.get("content"),
            "reasoning_text": msg.get("reasoning_content"),
            "gpu_after_generation": gpu_snapshot(),
            "log_tail": tail_text(LOG, 8000),
        })
except Exception as exc:
    report.update({
        "status": "FAIL",
        "failure_class": "HARNESS_OR_REQUEST_ERROR",
        "exception_type": type(exc).__name__,
        "exception": str(exc),
        "gpu_at_failure": gpu_snapshot(),
        "log_tail": tail_text(LOG) if LOG.exists() else "",
    })
finally:
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=10)
    OUT.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    print(json.dumps(report, indent=2, sort_keys=True))
    print(f"\nWROTE: {OUT}")
    print(f"LOG:   {LOG}")
